# CrewAI Flows

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app).

You will use CrewAI **Flows**: `@start`, `@listen`, Pydantic **state**, `@router` branching, **combinators** (`or_`, `and_`), and calling a **Crew** from inside a flow step.

In [ ]:
!pip install -q crewai crewai-tools pydantic

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## Basic flow: `@start` and `@listen`

Entry points are marked with `@start()`. Each `@listen(previous_step)` runs when that step finishes; its return value is passed in as the first argument (unless you use state-only patterns).

In [ ]:
from crewai.flow.flow import Flow, listen, start


class ResearchFlow(Flow):
    @start()
    def gather_topic(self):
        return "AI Agents in 2025"

    @listen(gather_topic)
    def research(self, topic):
        return f"Research results for: {topic}"

    @listen(research)
    def write_report(self, results):
        return f"Final report based on: {results}"


flow = ResearchFlow()
result = flow.kickoff()
print(result)

## Structured state with Pydantic

Subclass `Flow[YourState]` with a Pydantic `BaseModel`. Steps update `self.state` so downstream logic shares one typed object.

In [ ]:
from pydantic import BaseModel

from crewai.flow.flow import Flow, listen, start


class ResearchState(BaseModel):
    topic: str = ""
    findings: list[str] = []
    report: str = ""


class StatefulResearchFlow(Flow[ResearchState]):
    @start()
    def set_topic(self):
        self.state.topic = "AI Agents"
        return self.state.topic

    @listen(set_topic)
    def collect(self, topic):
        self.state.findings.append(f"Finding about {topic}")
        return self.state.findings

    @listen(collect)
    def finalize(self, findings):
        self.state.report = " | ".join(findings)
        return self.state.report


sflow = StatefulResearchFlow()
print(sflow.kickoff())
print(sflow.state.model_dump())

## Conditional routing with `@router`

After the routed method completes, `@router` returns a **string label**. The next step is the `@listen("label")` method whose label matches.

In [ ]:
from crewai.flow.flow import Flow, listen, router, start


class BranchingFlow(Flow):
    @start()
    def research(self):
        return "x" * 500

    @router(research)
    def decide_next(self, results):
        if len(results) > 1000:
            return "detailed_analysis"
        return "quick_summary"

    @listen("detailed_analysis")
    def analyze(self):
        return "detailed path"

    @listen("quick_summary")
    def summarize(self):
        return "quick path"


bflow = BranchingFlow()
print(bflow.kickoff())

## Flow step that calls a Crew

Inside any flow method, build or reuse a `Crew` and call `kickoff(inputs=...)`. Pass `CrewOutput` fields (for example `.raw`) to the next listener or write them into `self.state`.

In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai.flow.flow import Flow, listen, start

researcher = Agent(
    role="Researcher",
    goal="Summarize the topic briefly",
    backstory="You are concise and factual.",
    verbose=True,
)

research_task = Task(
    description="Research: {topic}. Output 3 short bullets.",
    expected_output="Three bullets.",
    agent=researcher,
)

research_crew = Crew(
    agents=[researcher],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True,
)


class CrewInsideFlow(Flow):
    @start()
    def run_research_crew(self):
        out = research_crew.kickoff(inputs={"topic": "AI agents"})
        return out.raw

    @listen(run_research_crew)
    def wrap_up(self, crew_output):
        preview = crew_output[:300] + ("..." if len(crew_output) > 300 else "")
        return f"Crew finished. Preview:\n{preview}"


cflow = CrewInsideFlow()
print(cflow.kickoff())

## `or_()` and `and_()` combinators

Use **`and_(step_a, step_b)`** inside `@listen(...)` when a method should run only after **all** listed steps complete (for example, after two parallel `@start` branches). Use **`or_(step_a, step_b)`** when **any** completion should trigger the listener. Exact listener signatures can vary by CrewAI version; check the [Flows](https://docs.crewai.com/concepts/flows) docs if your run errors on argument counts.

In [ ]:
from crewai.flow.flow import Flow, listen, start, and_


class ParallelMergeFlow(Flow):
    @start()
    def branch_a(self):
        return "A"

    @start()
    def branch_b(self):
        return "B"

    @listen(and_(branch_a, branch_b))
    def merge(self):
        return "both branches finished"


mflow = ParallelMergeFlow()
print(mflow.kickoff())

## Key takeaways

- **Flows** orchestrate **events** above crews: `@start` entry points, `@listen` chains, `@router` string labels for branches.
- Use **`Flow[PydanticModel]`** and **`self.state`** for typed, shared state across steps.
- Call **`crew.kickoff()`** inside flow methods like normal Python; forward **`.raw`** or structured fields to listeners.
- **`and_` / `or_`** express **all-of** or **any-of** triggers for listeners when multiple predecessors exist.
- Prefer a **plain Crew** for a single linear run; prefer a **Flow** for **multi-stage**, **conditional**, or **stateful** orchestration.